# Phase 5: Zone and Line Reasoning Exploration

This notebook explores visual zone/line authoring on a sample camera frame and evaluates:
- Polygon zone layout (`entrance_area`, `center_concourse`, `right_aisle`) and directed tripwire lines (`doorway_line`, `aisle_split_line`).
- Point-in-polygon (`zone_enter`, `zone_exit`) and directional line-crossing (`line_cross`, `A->B` / `B->A`) signals.
- Boundary flicker comparison between raw (`w=1`) and smoothed (`w=3`) trajectories.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while (
    not (PROJECT_ROOT / "pyproject.toml").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "cv_projects" / "05_zone_line_reasoning" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from define_scene_geometry import export_scene_geometry
from reason_zones_lines import (
    analyze_boundary_flicker,
    evaluate_scene_signals,
    load_scene_config,
    read_trajectories_csv,
)

In [ ]:
video_path = PROJECT_ROOT / "data" / "fallback_clips" / "people-detection.mp4"
scene_cfg_path = (
    PROJECT_ROOT
    / "cv_projects"
    / "05_zone_line_reasoning"
    / "configs"
    / "scene_config.yaml"
)
traj_csv = (
    PROJECT_ROOT
    / "cv_projects"
    / "04_trajectory_extraction"
    / "outputs"
    / "people_detection"
    / "trajectories.csv"
)
if video_path.exists() and scene_cfg_path.exists() and traj_csv.exists():
    cfg = load_scene_config(scene_cfg_path)
    trajs = read_trajectories_csv(traj_csv)
    signals = evaluate_scene_signals(trajs, cfg)
    for sig in signals:
        print(sig.to_row())
    flicker = analyze_boundary_flicker(signals, max_flicker_frames=5)
    print(json.dumps(flicker, indent=2))
    geom_report = export_scene_geometry(
        video_path,
        PROJECT_ROOT
        / "cv_projects"
        / "05_zone_line_reasoning"
        / "outputs"
        / "notebook_preview",
        scene_config_path=scene_cfg_path,
    )
    print(json.dumps(geom_report, indent=2))